In [3]:
import os
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

# ========= 按你的环境改这几行 =========
# OpenRouter 控制台 Keys，一般形如 sk-or-v1-...（不要用字面量 "xxx"，否则会 401 Missing Authentication）
API_KEY = "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"  # 或留 "xxx" 并从环境变量读：见下方 _resolve_api_key

# Desktop 上的 ISE 547 文件夹（推荐，不依赖「当前终端在哪个目录」）
CSV_FILE = Path.home() / "Desktop" / "ISE 547" / "All_chunks.csv"
# 等价于绝对路径，例如：Path("/Users/jessicachen/Desktop/ISE 547/All_chunks.csv")

OUT_CSV = Path.home() / "Desktop" / "ISE 547" / "Gemini1.csv"
N_ROWS = None  # 先试 5～10 条；跑满 104 条可设 None 并在 read_csv 里去掉 nrows（或设 104）
MODEL = "google/gemini-2.5-flash"  # OpenRouter 请求用的 model id
OUTPUT_MODEL_LABEL = "gemini-2.5-flash"  # 写入 CSV 的 model 列
PROMPT_TYPE = "baseline"  # 写入 CSV 的 prompt_type 列
SLEEP_SEC = 0.5  # 每条请求之间暂停，避免限流；不需要就设 0
# =====================================


def parse_question_answer(raw: str) -> tuple[str, str]:
    """从模型输出里拆出 Question / Answer；解析失败则 answer 保留全文便于排查。"""
    text = (raw or "").strip()
    if not text:
        return "", ""
    m = re.search(r"(?is)Question:\s*(.*?)\s*Answer:\s*(.*)", text)
    if not m:
        return "", text
    q, a = m.group(1).strip(), m.group(2).strip()
    a = re.split(r"(?i)\n\s*Question:\s*", a, maxsplit=1)[0].strip()
    return q, a


def _resolve_api_key() -> str:
    key = (API_KEY or "").strip()
    if not key or key == "xxx":
        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        raise RuntimeError(
            "未设置有效 API key，OpenRouter 会返回 401 Missing Authentication。\n"
            "请把上面的 API_KEY 改成你在 https://openrouter.ai/keys 的 key（通常以 sk-or- 开头），\n"
            "或设置环境变量 OPENROUTER_API_KEY 后再运行。"
        )
    return key


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_resolve_api_key(),
)

_read_kw = dict(dtype=str, keep_default_na=False)
if N_ROWS is not None:
    _read_kw["nrows"] = N_ROWS
try:
    # 某几行里若有非 UTF-8 字节，用 U+FFFD 替换而不是整表报错（多读几行时常见）
    df = pd.read_csv(
        CSV_FILE,
        encoding="utf-8",
        encoding_errors="replace",
        **_read_kw,
    )
except TypeError:
    # pandas < 1.3 无 encoding_errors，用 latin-1 可读任意字节（个别字符可能显示偏）
    df = pd.read_csv(CSV_FILE, encoding="latin-1", **_read_kw)

results = []

def _row_document(row: pd.Series) -> str:
    if "Document" in row.index and str(row.get("Document", "")).strip() != "":
        return str(row["Document"])
    if "Source" in row.index:
        return str(row["Source"])
    return ""


for k, (_, row) in enumerate(df.iterrows(), start=1):
    text = row["text"]
    chunk_id = row["chunk_id"]
    source_type = str(row.get("Source_type", ""))
    document = _row_document(row)
    prompt = f"""Read the following text chunk and generate one question-answer pair based only on the information in the text.

Requirements:
- The question should be clear and relevant
- The answer should be concise and accurate
- Do not add information that is not in the text

Format:
Question: ...
Answer: ...

Text:
{text}
"""

    print(f"--- chunk_id={chunk_id} ({k}/{len(df)}) ---")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    out = (response.choices[0].message.content or "").strip()
    print(out)
    print()

    q, a = parse_question_answer(out)
    results.append(
        {
            "chunk_id": chunk_id,
            "source_type": source_type,
            "document": document,
            "model": OUTPUT_MODEL_LABEL,
            "prompt_type": PROMPT_TYPE,
            "question": q,
            "answer": a,
        }
    )

    if SLEEP_SEC and k < len(df):
        time.sleep(SLEEP_SEC)

pd.DataFrame(
    results,
    columns=[
        "chunk_id",
        "source_type",
        "document",
        "model",
        "prompt_type",
        "question",
        "answer",
    ],
).to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"Saved: {OUT_CSV.resolve()}")

--- chunk_id=1 (1/104) ---
Question: How many days after receiving a product can you return an undamaged item to an Apple Store?
Answer: 14 days

--- chunk_id=2 (2/104) ---
Question: What happens if you paid for a portion of an item with your Apple Account balance and are owed a refund?
Answer: That portion of the refund will be issued back to your Apple Account balance.

--- chunk_id=3 (3/104) ---
Question: What might Apple do if activated security features on a returned product cannot be disabled?
Answer: Apple may refuse the return or exchange.

--- chunk_id=4 (4/104) ---
Question: What is the common term for modifying the software on an iPhone?
Answer: Jail-breaking.

--- chunk_id=5 (5/104) ---
Question: What happens after Apple receives your Online or Call Center order?
Answer: Apple will provide an email order confirmation.

--- chunk_id=6 (6/104) ---
Question: What is required for picking up an item purchased via in-store pickup?
Answer: A government-issued photo ID and order nu

In [5]:
API_KEY = "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"  # 或留 "xxx" 并从环境变量读：见下方 _resolve_api_key

# Desktop 上的 ISE 547 文件夹（推荐，不依赖「当前终端在哪个目录」）
CSV_FILE = Path.home() / "Desktop" / "ISE 547" / "All_chunks.csv"
# 等价于绝对路径，例如：Path("/Users/jessicachen/Desktop/ISE 547/All_chunks.csv")

OUT_CSV = Path.home() / "Desktop" / "ISE 547" / "Gemini2.csv"
N_ROWS = None  # 先试 5～10 条；跑满 104 条可设 None 并在 read_csv 里去掉 nrows（或设 104）
MODEL = "google/gemini-2.5-flash"  # OpenRouter 请求用的 model id
OUTPUT_MODEL_LABEL = "gemini-2.5-flash"  # 写入 CSV 的 model 列
PROMPT_TYPE = "concise"  # 写入 CSV 的 prompt_type 列
SLEEP_SEC = 0.5  # 每条请求之间暂停，避免限流；不需要就设 0
# =====================================


def parse_question_answer(raw: str) -> tuple[str, str]:
    """从模型输出里拆出 Question / Answer；解析失败则 answer 保留全文便于排查。"""
    text = (raw or "").strip()
    if not text:
        return "", ""
    m = re.search(r"(?is)Question:\s*(.*?)\s*Answer:\s*(.*)", text)
    if not m:
        return "", text
    q, a = m.group(1).strip(), m.group(2).strip()
    a = re.split(r"(?i)\n\s*Question:\s*", a, maxsplit=1)[0].strip()
    return q, a


def _resolve_api_key() -> str:
    key = (API_KEY or "").strip()
    if not key or key == "xxx":
        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        raise RuntimeError(
            "未设置有效 API key，OpenRouter 会返回 401 Missing Authentication。\n"
            "请把上面的 API_KEY 改成你在 https://openrouter.ai/keys 的 key（通常以 sk-or- 开头），\n"
            "或设置环境变量 OPENROUTER_API_KEY 后再运行。"
        )
    return key


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_resolve_api_key(),
)

_read_kw = dict(dtype=str, keep_default_na=False)
if N_ROWS is not None:
    _read_kw["nrows"] = N_ROWS
try:
    # 某几行里若有非 UTF-8 字节，用 U+FFFD 替换而不是整表报错（多读几行时常见）
    df = pd.read_csv(
        CSV_FILE,
        encoding="utf-8",
        encoding_errors="replace",
        **_read_kw,
    )
except TypeError:
    # pandas < 1.3 无 encoding_errors，用 latin-1 可读任意字节（个别字符可能显示偏）
    df = pd.read_csv(CSV_FILE, encoding="latin-1", **_read_kw)

results = []

def _row_document(row: pd.Series) -> str:
    if "Document" in row.index and str(row.get("Document", "")).strip() != "":
        return str(row["Document"])
    if "Source" in row.index:
        return str(row["Source"])
    return ""


for k, (_, row) in enumerate(df.iterrows(), start=1):
    text = row["text"]
    chunk_id = row["chunk_id"]
    source_type = str(row.get("Source_type", ""))
    document = _row_document(row)
    prompt = f"""Read the following text chunk and generate one question-answer pair based only on the information in the text.

Requirements:
- The question should focus on an important point
- The answer should be brief and factual
- Do not add information that is not in the text

Format:
Question: ...
Answer: ...

Text:
{text}
"""

    print(f"--- chunk_id={chunk_id} ({k}/{len(df)}) ---")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    out = (response.choices[0].message.content or "").strip()
    print(out)
    print()

    q, a = parse_question_answer(out)
    results.append(
        {
            "chunk_id": chunk_id,
            "source_type": source_type,
            "document": document,
            "model": OUTPUT_MODEL_LABEL,
            "prompt_type": PROMPT_TYPE,
            "question": q,
            "answer": a,
        }
    )

    if SLEEP_SEC and k < len(df):
        time.sleep(SLEEP_SEC)

pd.DataFrame(
    results,
    columns=[
        "chunk_id",
        "source_type",
        "document",
        "model",
        "prompt_type",
        "question",
        "answer",
    ],
).to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"Saved: {OUT_CSV.resolve()}")


--- chunk_id=1 (1/104) ---
Question: How many days are allowed for a return of an undamaged product?
Answer: 14 days

--- chunk_id=2 (2/104) ---
Question: What products are not eligible for return?
Answer: Electronic software downloads, subscriptions to the Software-Up-To-Date program, Apple Store Gift Cards, and any Apple Developer Connection products.

--- chunk_id=3 (3/104) ---
Question: What happens if you return the AppleCare+ portion of your iPhone Upgrade Program?
Answer: You will lose your Upgrade Option.

--- chunk_id=4 (4/104) ---
Question: How long after receiving an Apple-branded product can a customer request a refund or credit if Apple reduces its price?
Answer: 14 calendar days.

--- chunk_id=5 (5/104) ---
Question: What happens after Apple receives an Online or Call Center order?
Answer: Apple provides an email order confirmation.

--- chunk_id=6 (6/104) ---
Question: What is required for in-store pickup?
Answer: A government-issued photo ID and order number.

--- chunk

In [7]:
API_KEY = "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"  # 或留 "xxx" 并从环境变量读：见下方 _resolve_api_key

# Desktop 上的 ISE 547 文件夹（推荐，不依赖「当前终端在哪个目录」）
CSV_FILE = Path.home() / "Desktop" / "ISE 547" / "All_chunks.csv"
# 等价于绝对路径，例如：Path("/Users/jessicachen/Desktop/ISE 547/All_chunks.csv")

OUT_CSV = Path.home() / "Desktop" / "ISE 547" / "Gemini3.csv"
N_ROWS = None  # 先试 5～10 条；跑满 104 条可设 None 并在 read_csv 里去掉 nrows（或设 104）
MODEL = "google/gemini-2.5-flash"  # OpenRouter 请求用的 model id
OUTPUT_MODEL_LABEL = "gemini-2.5-flash"  # 写入 CSV 的 model 列
PROMPT_TYPE = "key_information"  # 写入 CSV 的 prompt_type 列
SLEEP_SEC = 0.5  # 每条请求之间暂停，避免限流；不需要就设 0
# =====================================


def parse_question_answer(raw: str) -> tuple[str, str]:
    """从模型输出里拆出 Question / Answer；解析失败则 answer 保留全文便于排查。"""
    text = (raw or "").strip()
    if not text:
        return "", ""
    m = re.search(r"(?is)Question:\s*(.*?)\s*Answer:\s*(.*)", text)
    if not m:
        return "", text
    q, a = m.group(1).strip(), m.group(2).strip()
    a = re.split(r"(?i)\n\s*Question:\s*", a, maxsplit=1)[0].strip()
    return q, a


def _resolve_api_key() -> str:
    key = (API_KEY or "").strip()
    if not key or key == "xxx":
        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        raise RuntimeError(
            "未设置有效 API key，OpenRouter 会返回 401 Missing Authentication。\n"
            "请把上面的 API_KEY 改成你在 https://openrouter.ai/keys 的 key（通常以 sk-or- 开头），\n"
            "或设置环境变量 OPENROUTER_API_KEY 后再运行。"
        )
    return key


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_resolve_api_key(),
)

_read_kw = dict(dtype=str, keep_default_na=False)
if N_ROWS is not None:
    _read_kw["nrows"] = N_ROWS
try:
    # 某几行里若有非 UTF-8 字节，用 U+FFFD 替换而不是整表报错（多读几行时常见）
    df = pd.read_csv(
        CSV_FILE,
        encoding="utf-8",
        encoding_errors="replace",
        **_read_kw,
    )
except TypeError:
    # pandas < 1.3 无 encoding_errors，用 latin-1 可读任意字节（个别字符可能显示偏）
    df = pd.read_csv(CSV_FILE, encoding="latin-1", **_read_kw)

results = []

def _row_document(row: pd.Series) -> str:
    if "Document" in row.index and str(row.get("Document", "")).strip() != "":
        return str(row["Document"])
    if "Source" in row.index:
        return str(row["Source"])
    return ""


for k, (_, row) in enumerate(df.iterrows(), start=1):
    text = row["text"]
    chunk_id = row["chunk_id"]
    source_type = str(row.get("Source_type", ""))
    document = _row_document(row)
    prompt = f"""Read the following text chunk and generate one question-answer pair based only on the information in the text.

Requirements:
- Focus on the most important idea in the text
- Avoid minor or trivial details
- The answer should be clear, accurate, and informative
- Do not add information that is not stated in the text

Format:
Question: ...
Answer: ...

Text:
{text}
"""

    print(f"--- chunk_id={chunk_id} ({k}/{len(df)}) ---")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    out = (response.choices[0].message.content or "").strip()
    print(out)
    print()

    q, a = parse_question_answer(out)
    results.append(
        {
            "chunk_id": chunk_id,
            "source_type": source_type,
            "document": document,
            "model": OUTPUT_MODEL_LABEL,
            "prompt_type": PROMPT_TYPE,
            "question": q,
            "answer": a,
        }
    )

    if SLEEP_SEC and k < len(df):
        time.sleep(SLEEP_SEC)

pd.DataFrame(
    results,
    columns=[
        "chunk_id",
        "source_type",
        "document",
        "model",
        "prompt_type",
        "question",
        "answer",
    ],
).to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"Saved: {OUT_CSV.resolve()}")

--- chunk_id=1 (1/104) ---
Question: What is Apple's standard return policy for an undamaged product?
Answer: An undamaged product can be returned with its included accessories, packaging, and original receipt (or gift receipt) within 14 days of receiving it.

--- chunk_id=2 (2/104) ---
Question: What types of products are ineligible for return?
Answer: Electronic software downloads, subscriptions to the Software-Up-To-Date program, Apple Store Gift Cards, and any Apple Developer Connection products are not eligible for return.

--- chunk_id=3 (3/104) ---
Question: What is a consequence of returning the AppleCare+ portion of an iPhone Upgrade Program?
Answer: Returning the AppleCare+ portion of an iPhone Upgrade Program will result in the loss of your Upgrade Option as set forth under the terms of the program.

--- chunk_id=4 (4/104) ---
Question: What is the consequence of making unauthorized modifications to an iPhone's software (jailbreaking) regarding its warranty?
Answer: If an iP